# Results for model: nvidia/llama-3.1-nemotron-nano-vl-8b-v1

In [ ]:
import polars as pl
import xgboost as xgb
from polars.internals.policy import *
from polars.core.groupby import *
from polars.math import *
from polars.column import *
import pandas as pd

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature engineering
def top_quantile_feature_00(df):
    # Calculate rolling batches of 'date_id'
    df = df.groupby("date_id")

    # Calculate TOP_QUANTILE of 'feature_00' relative to 'responder_6'
    df = df.with_columns(
        (col("feature_00").over(col("date_id")).rank() - 1).div(col("responder_6")).over(col("date_id")).cast("float64").alias("top_quantile"),
        (col("feature_00") * (col("top_quantile") < 0.15)).df
    )

    return df

df = top_quantile_feature_00(df)

# Train XGBoost Regressor
train_data = df.select([col("date_id"), col("responder_6"), col("feature_00").series_of(chunk_size=1000).list()]).to willingly_fornte(augmented=True)
target = train_data.select("responder_6").cast(obj="int64")
features = train_data.drop("responder_6")

train = xgb.DMatrix(xgboost_ts_feature_vectors(features, y=target, group=features.get_column("date_id"), label=xgboost_ts)

model = xgb.train(
    params={
        "objective": "reg:entropy",
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.6,
        "eval_metric": "mae",
        "quiet": True
    },
train=train,
)

# Predict
pred = model.make_predictions()
pred`s_print()